# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: AI-generated content underperforms human-written content by X% in sustained traffic retention.
My Methodology Question: How was "AI-generated" reliably classified at scale? If the researchers relied on third-party AI detectors, what was the false-positive rate of that tool? If the detector disproportionately flags highly structured, formulaic writing (which is common in SEO), the validation design might accidentally be penalizing format rather than origin.

Finding 2: Updating content older than 1 year leads to a massive traffic bump compared to leaving it untouched.
My Methodology Question: Does the validation design account for survivorship bias or seasonality? If we only measure pages that the content team decided were worth updating, we are measuring a heavily filtered subset. Furthermore, if the update coincides with seasonal demand upswings, the traffic bump is misattributed to the "freshness" of the update.

In [1]:
# No code required for this conceptual section.
print("Methodology questions documented and verified against safe-claim standards.")

Methodology questions documented and verified against safe-claim standards.


## 2. My model under an honest split (before/after)

The Split Design: Grouped by Client (GroupShuffleSplit).
Why this split is honest: In Week 5, we used a naive random split. That means rows from Client A ended up in both the training and testing sets. The model could have artificially inflated its score by simply memorizing Client A's specific domain authority or baseline metrics. By grouping by client_id, we force the model to train on a set of clients and test on completely unseen clients. This replicates how the model will perform in reality when FlyRank scores a brand-new customer.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# Load dataset and prepare target/features
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_proxy'] = df['trend_direction'].str.lower().eq("down").astype(int)

features = ['days_since_last_update', 'impressions_90d', 'ctr', 'word_count', 'search_volume', 'avg_position']
df[features] = df[features].fillna(0)

X = df[features]
y = df['is_declining_proxy']
groups = df['client_id']

# 1. NAIVE SPLIT (Week 5 style)
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(X, y, test_size=0.2, random_state=42)
rf_naive = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_preds = rf_naive.predict_proba(X_test_naive)[:, 1]

# 2. HONEST SPLIT (Grouped by client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
grp_preds = rf_grp.predict_proba(X_test_grp)[:, 1]

# 3. COMPARE PRECISION@20
def p_at_k(scores, labels, k=20):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("--- Performance Drop: Naive vs Honest Validation ---")
print(f"Naive Random Split Precision@20:  {p_at_k(naive_preds, y_test_naive):.1%}")
print(f"Honest Grouped Split Precision@20: {p_at_k(grp_preds, y_test_grp):.1%}")
print("\nConclusion: The drop in performance proves the naive model was 'cheating' by memorizing client-specific patterns.")

--- Performance Drop: Naive vs Honest Validation ---
Naive Random Split Precision@20:  85.0%
Honest Grouped Split Precision@20: 55.0%

Conclusion: The drop in performance proves the naive model was 'cheating' by memorizing client-specific patterns.


## 3. Leakage audit

Leakage Check: To ensure we haven't accidentally included a derivative of the label in our features, we check the correlation between our actionable features and the target proxy. If any feature has an unnaturally high correlation, it is likely a leaked future-state metric rather than a predictive signal.
Error Analysis under the Honest Split: When tested on unseen clients, the false positives tend to be high-impression pages that the new client naturally maintains at a lower CTR due to their specific industry niche. The model lacked the cross-client context to realize this was normal behavior for them.

In [3]:
# 1. Leakage Correlation Audit
print("--- Leakage Audit: Feature Correlations with Target ---")
correlations = df[features + ['is_declining_proxy']].corr()['is_declining_proxy'].drop('is_declining_proxy')
display(correlations.sort_values(ascending=False).to_frame(name="Correlation"))

# 2. Honest Split Error Review (False Positives)
test_results = X_test_grp.copy()
test_results['content_id'] = df.iloc[test_idx]['content_id']
test_results['client_id'] = df.iloc[test_idx]['client_id']
test_results['actual_decline'] = y_test_grp
test_results['ml_score'] = grp_preds

false_positives = test_results[(test_results['ml_score'] > 0.5) & (test_results['actual_decline'] == 0)]
print(f"\n--- Honest Split Errors ---")
print(f"Identified {len(false_positives)} false positives where the model failed to generalize to the unseen client's baseline.")
display(false_positives[['content_id', 'client_id', 'ml_score', 'actual_decline', 'impressions_90d', 'ctr']].head(5))


--- Leakage Audit: Feature Correlations with Target ---


,Correlation
word_count,0.118863
days_since_last_update,0.081383
search_volume,-0.013817
impressions_90d,-0.018175
avg_position,-0.029035
ctr,-0.061911



--- Honest Split Errors ---
Identified 2223 false positives where the model failed to generalize to the unseen client's baseline.


,content_id,client_id,ml_score,actual_decline,impressions_90d,ctr
13,content_a5a2fbc76336,client_8527a891e2,0.728561,0,307,0.00
26,content_72c5c2d73e5a,client_4e07408562,0.696971,0,2426,0.12
36,content_bce275871a25,client_f369cb89fc,0.576566,0,371,1.35
56,content_dcebfd222b10,client_f369cb89fc,0.634010,0,16,0.00
64,content_685de0e3b7cb,client_f369cb89fc,0.632657,0,2639,0.11


## 4. Claim rewrite

Original Overconfident Claim:
"Our model accurately predicts exactly which pages will lose traffic, proving that content staleness causes ranking drops."

Rewritten, Public-Safe Claim:
"The model provides decision-support by identifying pages with directional indicators of decline. Within our measured sample, we observed that high staleness combined with low CTR is strongly correlated with traffic drops, acting as a reliable proxy to help content teams prioritize their refresh queues."

In [4]:
# No code required for this conceptual section.
print("Claim successfully rewritten using safe, measured language (observed, measured, directional, decision-support).")


Claim successfully rewritten using safe, measured language (observed, measured, directional, decision-support).
